# Retail Gold ETL — AWS Glue / PySpark

Notebook converted from the exported AWS Glue job `retail-gold-etl`.

## Exported Glue job configuration
- **Glue version:** 6.0
- **Worker type:** G.1X
- **Workers:** 2
- **Timeout:** 20 minutes
- **Job bookmark:** job-bookmark-disable

> The code cells preserve the script embedded in the Glue export. Execution requires a compatible Spark/AWS Glue environment and access to the referenced S3 paths.


## ETL code


In [ ]:
import sys
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql import Row
from awsglue.context import GlueContext
from awsglue.job import Job

# --------------------------------------------------


## INITIALIZATION


In [ ]:
# 1. INITIALIZATION
# --------------------------------------------------

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

job = Job(glueContext)
job.init("retail-gold-etl",{})

BUCKET = "tus-retail-clothing-store-datasets"

SILVER_PATH = f"s3://{BUCKET}/silver/"
GOLD_PATH = f"s3://{BUCKET}/gold/"

# --------------------------------------------------


## READ SILVER DATA


In [ ]:
# 2. READ SILVER DATA
# --------------------------------------------------

products = spark.read.parquet(SILVER_PATH + "products/")
brands = spark.read.parquet(SILVER_PATH + "brands/")
category = spark.read.parquet(SILVER_PATH + "category/")
customers = spark.read.parquet(SILVER_PATH + "customers/")
calendar = spark.read.parquet(SILVER_PATH + "date/")
order_items = spark.read.parquet(SILVER_PATH + "order_items/")

# --------------------------------------------------


## GOLD DIMENSION: PRODUCTS


In [ ]:
# 3. GOLD DIMENSION: PRODUCTS
# --------------------------------------------------

# JOINING BRANDS AND CATEGORY TABLE

brands_categories = (
    brands.alias("b").join(category.alias("c"), F.col("b.category_code") == F.col("c.category_code"), "inner")
    .select(
        F.col("b.brand_name"),
        F.col("b.brand_code"),
        F.col("c.category_name"),
        F.col("c.category_code")
        )
    )

# Then left joins the mapping to products using brand_code.

dim_products = (
    products.alias("p")
    .join(
        brands_categories.alias("bc"),
        F.col("p.brand_code") == F.col("bc.brand_code"),
        "left"
    )
    .select(
        F.col("p.product_id"),
        F.col("p.sku"),
        F.col("p.category_code"),
        F.coalesce(
            F.col("bc.category_name"),
            F.lit("Not Available")
        ).alias("category_name"),
        F.col("p.brand_code"),
        F.coalesce(
            F.col("bc.brand_name"),
            F.lit("Not Available")
        ).alias("brand_name"),
        F.col("p.color"),
        F.col("p.size"),
        F.col("p.material"),
        F.col("p.weight_grams"),
        F.col("p.length_cm"),
        F.col("p.width_cm"),
        F.col("p.height_cm"),
        F.col("p.rating_count"),
        F.col("p.source_file").alias("file_name"),
        F.col("p.ingestion_timestamp").alias("ingest_timestamp")
    )
)

dim_products.write.mode("overwrite").parquet(
    GOLD_PATH + "dim_products/"
)

# --------------------------------------------------


## GOLD DIMENSION: CUSTOMERS


In [ ]:
# 4. GOLD DIMENSION: CUSTOMERS
# --------------------------------------------------

# Original country-state-region mapping.

country_state_map = {
    "India": {
        "MH": "West", "GJ": "West", "RJ": "West",
        "KA": "South", "TN": "South", "TS": "South",
        "AP": "South", "KL": "South",
        "UP": "North", "WB": "North", "DL": "North"
    },
    "Australia": {
        "VIC": "SouthEast", "WA": "West",
        "NSW": "East", "QLD": "NorthEast"
    },
    "United Kingdom": {
        "ENG": "England", "WLS": "Wales",
        "NIR": "Northern Ireland", "SCT": "Scotland"
    },
    "United States": {
        "MA": "NorthEast", "FL": "South",
        "NJ": "NorthEast", "CA": "West",
        "NY": "NorthEast", "TX": "South"
    },
    "United Arab Emirates": {
        "AUH": "Abu Dhabi", "DU": "Dubai",
        "SHJ": "Sharjah"
    },
    "Singapore": {
        "SG": "Singapore"
    },
    "Canada": {
        "BC": "West", "AB": "West",
        "ON": "East", "QC": "East",
        "NS": "East", "IL": "Other"
    }
}

#Converted the dict into 1D
mapping_rows = []

for country, states in country_state_map.items():
    for state_code, region in states.items():
        mapping_rows.append(
            Row(
                country=country,
                state=state_code,
                region=region
            )
        )

region_mapping = spark.createDataFrame(mapping_rows)

dim_customers = (
    customers
    .join(
        region_mapping,
        on=["country", "state"],
        how="left"
    )
    .fillna({"region": "Other"})
)

dim_customers.write.mode("overwrite").parquet(
    GOLD_PATH + "dim_customers/"
)

# --------------------------------------------------


## GOLD DIMENSION: DATE


In [ ]:
# 5. GOLD DIMENSION: DATE
# --------------------------------------------------


dim_date = (
    calendar
    .withColumn(
        "date_id",
        F.date_format(F.col("date"), "yyyyMMdd").cast("int")
    )
    .withColumn(
        "month_name",
        F.date_format(F.col("date"), "MMMM")
    )
    .withColumn(
        "is_weekend",
        F.when(
            F.col("day_name").isin("Saturday", "Sunday"), 1
        ).otherwise(0)
    )
    .select(
        "date_id",
        "date",
        "year",
        "month_name",
        "day_name",
        "is_weekend",
        "quarter",
        "week",
        F.col("ingestion_timestamp").alias("_ingested_at"),
        F.col("source_file").alias("_source_file")
    )
)

dim_date.write.mode("overwrite").parquet(
    GOLD_PATH + "dim_date/"
)

# --------------------------------------------------


## GOLD FACT: ORDER ITEMS


In [ ]:
# 6. GOLD FACT: ORDER ITEMS
# --------------------------------------------------

fact = (
    order_items
    .withColumn(
        "gross_amount",
        F.col("quantity") * F.col("unit_price")
    )
    .withColumn(
        "discount_amount",
        F.ceil(
            F.col("gross_amount") *
            (F.col("discount_pct") / 100.0)
        )
    )
    .withColumn(
        "sale_amount",
        F.col("gross_amount")
        - F.col("discount_amount")
        + F.col("tax_amount")
    )
    .withColumn(
        "date_id",
        F.date_format(F.col("dt"), "yyyyMMdd").cast("int")
    )
    .withColumn(
        "coupon_flag",
        F.when(F.col("coupon_code").isNotNull(), 1)
         .otherwise(0)
    )
)

# Original fixed exchange rates (as of 2025-10-15).
fx_rates = {
    "INR": 1.00,
    "AED": 24.18,
    "AUD": 57.55,
    "CAD": 62.93,
    "GBP": 117.98,
    "SGD": 68.18,
    "USD": 88.29
}

rates_df = spark.createDataFrame(
    [(k, float(v)) for k, v in fx_rates.items()],
    ["currency", "inr_rate"]
)

fact = (
    fact
    .join(
        rates_df,
        rates_df.currency == F.upper(
            F.trim(F.col("unit_price_currency"))
        ),
        "left"
    )
    .withColumn(
        "sale_amount_inr",
        F.ceil(F.col("sale_amount") * F.col("inr_rate"))
    )
)

# Select the exact output columns from the original notebook.
fact_gold = fact.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sale_amount").alias("net_amount"),
    F.col("sale_amount_inr").alias("net_amount_inr")
)

fact_gold.write.mode("overwrite").parquet(
    GOLD_PATH + "fact_order_items/"
)

# --------------------------------------------------


## VALIDATION COUNTS IN CLOUDWATCH LOGS


In [ ]:
# 7. VALIDATION COUNTS IN CLOUDWATCH LOGS
# --------------------------------------------------

print("GOLD LAYER ROW COUNTS")

print("dim_products:", dim_products.count())
print("dim_customers:", dim_customers.count())
print("dim_date:", dim_date.count())
print("fact_order_items:", fact_gold.count())

job.commit()
